# SIRUS entraîné sur les seuls produits à plusieurs candidats

**Question.** Le modèle SIRUS en production (`mlflow-artifacts:/15/c0c0ab52…`, entraîné sur le run
`2026-09-08/codif-c9vjm`) a appris sur **toutes** les lignes candidat, y compris les produits à
candidat unique, où il n'y a rien à choisir. Ces produits sont désormais **routés** en amont
(`scorer.split_single_candidates`) : ils reçoivent leur unique code sans passer par le modèle.
Un modèle entraîné uniquement sur les produits à ≥ 2 candidats fait-il mieux là où SIRUS tranche
réellement ?

**Protocole.**
- Même run, même découpage 80/20 **par produit** (graine 42), même hyperparamètres (20 règles, profondeur 2, graine 42).
- Le filtre multi-candidats est appliqué **après** le découpage : le test porte sur les mêmes produits pour les deux modèles.
- Comparaison des deux **modèles d'évaluation** (ajustés sur les 80 %) sur le test 20 %. Les modèles livrés (100 %)
  ont vu le test : ils ne servent qu'à lire les règles.

**Aucune écriture distante.** Le notebook lit S3 (sorties du run) et MLflow (artefacts de l'ancien modèle),
et n'écrit que dans `artifacts/codif-c9vjm-multi/` en local. Pas de `mlflow.start_run`, pas de `finalize`.

**Lancement** (depuis `reconcile-sirus/`, avec les identifiants S3 et `MLFLOW_TRACKING_URI` du service) :

```bash
Rscript R/install_deps.R          # une fois par service : sirus patché
uv run --with ipykernel --with jupyter jupyter lab notebooks/sirus_multi_candidats.ipynb
```

L'ajustement R prend quelques minutes.

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Le notebook tourne depuis reconcile-sirus/notebooks/ ou reconcile-sirus/.
MODULE = Path.cwd() if (Path.cwd() / "main.py").exists() else Path.cwd().parent
assert (MODULE / "R" / "fit_sirus.R").exists(), f"dossier reconcile-sirus introuvable depuis {Path.cwd()}"
sys.path.insert(0, str(MODULE))

from reconcile_llm import load_all_observations
from src.candidates import FEATURES, build_candidate_table, features_sha256, reconcile_population
from src.scorer import load_rules, pick_best, route_and_score, score
from src.train import build_calibration, keep_multi_candidates, split_by_product, verify_scorer_against_r

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [2]:
# --- Configuration ------------------------------------------------------------
RUN = "2026-09-08/codif-c9vjm"
ROOT = f"s3://projet-budget-famille/data/workflow_runs/{RUN}"   # LECTURE seule
ANCIEN_MODELE_URI = "mlflow-artifacts:/15/c0c0ab52a6e94a4684691d2e1ef10127/artifacts/model/"  # argo/params.yaml

NUM_RULE, MAX_DEPTH, SEED = 20, 2, 42
SPLIT_FRAC, SPLIT_SEED = 0.8, 42

OUT = MODULE / "artifacts" / "codif-c9vjm-multi"      # sorties locales du nouveau modèle
ANCIEN = MODULE / "artifacts" / "codif-c9vjm-ancien"  # copie locale de l'ancien modèle
OUT.mkdir(parents=True, exist_ok=True)
ANCIEN.mkdir(parents=True, exist_ok=True)

## 1. Table candidat

Mêmes entrées que `train.sh` (chemins de `contracts.yaml`), mêmes fonctions que `main.py build-table`.

In [3]:
merged = load_all_observations(
    lcs_path=f"{ROOT}/classify-lcs/raw_test_LCS.parquet",
    rag_path=f"{ROOT}/classify-rag-notices/predictions.parquet",
    ttc_path=f"{ROOT}/classify-ttc/predictions.parquet",
    rag_annotations_path=f"{ROOT}/classify-rag-annotations/predictions.parquet",
    mapping_path=f"{ROOT}/prune-codes/mapping_lvl4.parquet",
)
table, diag = build_candidate_table(merged)
tocodify = set(pd.read_parquet(f"{ROOT}/classify-regex/raw_test_without_regex.parquet", columns=["id"])["id"])
diag = reconcile_population(table, tocodify, diag)
table = split_by_product(table, frac=SPLIT_FRAC, seed=SPLIT_SEED)
print(f"{table['id'].nunique()} produits, {len(table)} candidats, "
      f"{diag['n_products_missing_from_lcs_base']} produit(s) perdu(s) par la base LCS")

14695 produits, 27604 candidats, 0 produit(s) perdu(s) par la base LCS


**Contrôles de reproductibilité.** La table doit être celle qui a servi à l'ancien modèle
(`artifacts/codif-c9vjm/features.parquet`, si présent localement), et la feature engineering
(`src/candidates.py`) doit être celle de l'ancien modèle — sinon la comparaison ne vaut rien.

In [4]:
# Artefacts de l'ancien modèle : téléchargement MLflow en LECTURE (aucun run créé).
import mlflow

mlflow.artifacts.download_artifacts(artifact_uri=ANCIEN_MODELE_URI, dst_path=str(ANCIEN))
ancien_dir = next(p.parent for p in ANCIEN.rglob("rules_eval.json"))
ancien_meta = json.loads((ancien_dir / "rules.json").read_text(encoding="utf-8"))
print("artefacts de l'ancien modèle :", sorted(p.name for p in ancien_dir.iterdir()))

assert ancien_meta["features_sha256"] == features_sha256(), "src/candidates.py a changé depuis l'ancien modèle"
print("features_sha256 identique à celui de l'ancien modèle ✔")

reference = MODULE / "artifacts" / "codif-c9vjm" / "features.parquet"
if reference.exists():
    ref = pd.read_parquet(reference)
    cols = ["id", "code_candidat", *FEATURES, "correcte", "split"]
    a = table[cols].astype({"code_candidat_n1": "string"}).reset_index(drop=True)
    b = ref[cols].astype({"code_candidat_n1": "string"}).reset_index(drop=True)
    pd.testing.assert_frame_equal(a, b, check_dtype=False)
    print("table identique à artifacts/codif-c9vjm/features.parquet (split compris) ✔")
else:
    print("artifacts/codif-c9vjm/features.parquet absent : contrôle de table sauté")

print("entraînement de l'ancien modèle :", {k: ancien_meta["training"][k] for k in ("run_id", "n_products", "n_candidates")})
assert ancien_meta["training"]["n_candidates"] == len(table), "l'ancien modèle n'a pas été entraîné sur cette table"

artefacts de l'ancien modèle : ['rules.json', 'rules_eval.json', 'rules_printed.txt']
features_sha256 identique à celui de l'ancien modèle ✔
table identique à artifacts/codif-c9vjm/features.parquet (split compris) ✔
entraînement de l'ancien modèle : {'run_id': 'codif-c9vjm', 'n_products': 14695, 'n_candidates': 27604}


## 2. Population : ce que le filtre écarte

In [5]:
n_cand = table.groupby("id")["id"].transform("size")
cas = np.select(
    [(n_cand == 1) & (table["nb_votants"] == 4), n_cand == 1],
    ["1 candidat, 4 votants (unanimité)", "1 candidat, < 4 votants (abstentions)"],
    "≥ 2 candidats",
)
population = (
    table.assign(cas=cas)
    .groupby("cas")
    .agg(produits=("id", "nunique"), lignes=("id", "size"), taux_correct=("correcte", "mean"))
)
population["part_lignes"] = population["lignes"] / population["lignes"].sum()
population.round(3)

,produits,lignes,taux_correct,part_lignes
cas,,,,
"1 candidat, 4 votants (unanimité)",5093,5093,0.975,0.185
"1 candidat, < 4 votants (abstentions)",1267,1267,0.959,0.046
≥ 2 candidats,8335,21244,0.326,0.770


## 3. Entraînement du nouveau modèle (multi-candidats seulement)

`keep_multi_candidates` est appliqué après `split_by_product` ; puis `R/fit_sirus.R`, inchangé,
ajuste le modèle d'évaluation (80 %) et le modèle livré (100 %).

In [6]:
multi = keep_multi_candidates(table)
multi.to_parquet(OUT / "features.parquet", index=False)
print(f"{multi['id'].nunique()} produits, {len(multi)} candidats "
      f"({(multi['split'] == 'train').sum()} train / {(multi['split'] == 'test').sum()} test), "
      f"taux de base {multi['correcte'].mean():.3f} contre {table['correcte'].mean():.3f} avant filtre")

8335 produits, 21244 candidats (17022 train / 4222 test), taux de base 0.326 contre 0.475 avant filtre


In [7]:
cmd = [
    "Rscript", "R/fit_sirus.R",
    f"--features={OUT / 'features.parquet'}", f"--out-dir={OUT}",
    f"--num-rule={NUM_RULE}", f"--max-depth={MAX_DEPTH}", f"--seed={SEED}",
    f"--features-sha256={features_sha256()}",
]
res = subprocess.run(cmd, cwd=MODULE, capture_output=True, text=True)
print(res.stdout[-3000:])
if res.returncode != 0:
    print(res.stderr[-3000:])
    raise RuntimeError("échec de fit_sirus.R")

Features : 21244 candidats (17022 train / 4222 test), 8335 produits, 15 modalités de division

Ajustement du modèle d'évaluation (80 %)...
  20 règles retenues, 5000 arbres

Réajustement du modèle livré (100 %)...
  20 règles retenues, 4000 arbres

Écrit dans /home/onyxia/work/codif-coicop-bdf/reconcile-sirus/artifacts/codif-c9vjm-multi :
  rules.json rules_eval.json proba_eval_R.csv rules_printed.txt



In [8]:
# Auto-contrôle (comme `finalize`) : le scorer Python reproduit sirus.predict au bit près.
test_multi = multi[multi["split"] == "test"].reset_index(drop=True)
nouveau_eval = load_rules(OUT / "rules_eval.json")
nouveau_final = load_rules(OUT / "rules.json")
assert verify_scorer_against_r(nouveau_eval, test_multi, OUT / "proba_eval_R.csv"), "le scorer Python diverge de R"
print("auto-contrôle R ↔ Python OK ✔")
print(f"règles retenues : modèle d'évaluation {len(nouveau_eval.rules)}, modèle livré {len(nouveau_final.rules)}")

auto-contrôle R ↔ Python OK ✔
règles retenues : modèle d'évaluation 20, modèle livré 20


## 4. Comparaison sur les produits test où SIRUS tranche vraiment

Les deux modèles d'évaluation (80 %) scorent **les mêmes candidats** : ceux des produits test à ≥ 2 candidats.

In [9]:
ancien_eval = load_rules(ancien_dir / "rules_eval.json")
ancien_final = load_rules(ancien_dir / "rules.json")

p_ancien = score(ancien_eval, test_multi[FEATURES])
p_nouveau = score(nouveau_eval, test_multi[FEATURES])
cal = {"ancien": build_calibration(test_multi, p_ancien), "nouveau": build_calibration(test_multi, p_nouveau)}

synthese = pd.DataFrame({
    nom: {
        "produits test multi": c["metrics"]["n_test_products"],
        "accuracy produit (argmax)": c["metrics"]["accuracy_product"],
        "borne haute (bon code proposé)": c["metrics"]["upper_bound"],
        "accuracy candidat (seuil 0,5)": c["metrics"]["accuracy_candidate"],
        "proba min": c["proba_min"],
        "proba max": c["proba_max"],
    }
    for nom, c in cal.items()
})
synthese["écart"] = synthese["nouveau"] - synthese["ancien"]
synthese.round(4)

,ancien,nouveau,écart
produits test multi,1662.0000,1662.0000,0.0000
accuracy produit (argmax),0.6721,0.6558,-0.0162
borne haute (bon code proposé),0.8394,0.8394,0.0000
"accuracy candidat (seuil 0,5)",0.8335,0.7932,-0.0403
proba min,0.3465,0.2246,-0.1219
proba max,0.6679,0.6043,-0.0635


### Fiabilité à volume égal

Les deux modèles n'ont pas la même plage de scores : un même seuil ne veut pas dire la même chose.
On compare donc la fiabilité des **X % de produits les mieux notés** par chaque modèle
(le volume qu'on laisserait passer en code automatique).

In [10]:
def fiabilite_par_volume(c, volumes=np.arange(0.1, 1.01, 0.1)):
    d = pd.DataFrame({"proba": c["proba"], "correct": c["correct"]}).sort_values("proba", ascending=False)
    return pd.Series({f"{v:.0%}": d["correct"].iloc[: max(1, round(v * len(d)))].mean() for v in volumes})

volume = pd.DataFrame({nom: fiabilite_par_volume(c) for nom, c in cal.items()})
volume["écart"] = volume["nouveau"] - volume["ancien"]
volume.index.name = "volume codé automatiquement"
volume.round(4)

,ancien,nouveau,écart
volume codé automatiquement,,,
10%,0.9940,0.9880,-0.0060
20%,0.9458,0.9458,0.0000
30%,0.9379,0.9339,-0.0040
40%,0.8992,0.9068,0.0075
50%,0.8736,0.8748,0.0012
60%,0.8245,0.8516,0.0271
70%,0.7997,0.8065,0.0069
80%,0.7534,0.7549,0.0015
90%,0.7106,0.6918,-0.0187


In [11]:
# Balayage par seuil (indicatif : les plages de score diffèrent).
def sweep(c):
    s = pd.DataFrame(c["threshold_sweep"]).set_index("seuil")
    return s.loc[s.index.isin(np.round(np.arange(0.35, 0.86, 0.05), 2)), ["volume", "reliability", "n"]]

pd.concat({nom: sweep(c) for nom, c in cal.items()}, axis=1).round(3)

ancien                   nouveau                  
      volume reliability     n  volume reliability     n
seuil                                                   
0.35   0.937       0.707  1558   0.774       0.774  1287
0.40   0.868       0.720  1443   0.644       0.831  1070
0.45   0.769       0.772  1278   0.576       0.864   958
0.50   0.710       0.801  1180   0.366       0.929   608
0.55   0.502       0.874   834   0.194       0.947   323
0.60   0.362       0.905   602   0.107       0.989   178
0.65   0.136       0.987   226   0.000         NaN     0
0.70   0.000         NaN     0   0.000         NaN     0
0.75   0.000         NaN     0   0.000         NaN     0
0.80   0.000         NaN     0   0.000         NaN     0
0.85   0.000         NaN     0   0.000         NaN     0

In [12]:
# Calibration : accuracy par tranche de score.
pd.concat(
    {nom: pd.DataFrame(c["bins"]).assign(tranche=lambda d: d["lo"].round(1).astype(str) + "–" + d["hi"].round(1).astype(str))
          .set_index("tranche")[["n", "pct_correct"]] for nom, c in cal.items()},
    axis=1,
).round(3)

ancien             nouveau            
             n pct_correct       n pct_correct
tranche                                       
0.3–0.4  219.0       0.356     435       0.345
0.4–0.5  263.0       0.357     462       0.701
0.5–0.6  578.0       0.692     430       0.905
0.6–0.7  602.0       0.905     178       0.989
0.2–0.3    NaN         NaN     157       0.325

### Désaccords entre les deux modèles

In [13]:
choix = {}
for nom, p in (("ancien", p_ancien), ("nouveau", p_nouveau)):
    best = pick_best(test_multi, p).merge(test_multi[["id", "code_candidat", "correcte"]],
                                          left_on=["id", "sirus_code"], right_on=["id", "code_candidat"])
    choix[nom] = best.set_index("id")[["sirus_code", "sirus_proba", "correcte"]]
cmp = choix["ancien"].join(choix["nouveau"], lsuffix="_ancien", rsuffix="_nouveau")
differents = cmp[cmp["sirus_code_ancien"] != cmp["sirus_code_nouveau"]]
print(f"{len(differents)} produits sur {len(cmp)} changent de code ({len(differents) / len(cmp):.1%})")
pd.crosstab(differents["correcte_ancien"].map({1: "ancien juste", 0: "ancien faux"}),
            differents["correcte_nouveau"].map({1: "nouveau juste", 0: "nouveau faux"}))

260 produits sur 1662 changent de code (15.6%)


correcte_nouveau,nouveau faux,nouveau juste
correcte_ancien,,
ancien faux,135,49
ancien juste,76,0


In [14]:
# Exemples : produits où un seul des deux modèles a raison.
libelle = next((c for c in ("product", "libelle", "product_name", "description") if c in merged.columns), None)
ex = differents[differents["correcte_ancien"] != differents["correcte_nouveau"]].join(
    merged.set_index("id")[[c for c in (libelle, "code_lvl4") if c]]
)
ex.sample(min(15, len(ex)), random_state=0)

,sirus_code_ancien,sirus_proba_ancien,correcte_ancien,sirus_code_nouveau,sirus_proba_nouveau,correcte_nouveau,code_lvl4
id,,,,,,,
095b4b17-5f14-4e5e-a9b2-afd7ee5498db,01.1.9.1,0.346524,1,98.1.1,0.336935,0,01.1.9.1
61a268e8-548e-4d6e-9325-d64e506690d9,01.1.6.5,0.435810,0,01.1.6,0.336935,1,01.1.6
511b421d-3ef8-4059-91cd-52522e8b69fb,07.1.3,0.496750,0,07.2.3,0.364777,1,07.2.3
2a240bf2-8562-4f3a-919d-67c1c6a7a7f3,01.1.7.5,0.527854,1,01.1.6,0.336935,0,01.1.7.5
85ea927c-b8b2-46b1-b56a-fc3cd473d11c,01.1.6.8,0.496750,1,01.1.7.1,0.336935,0,01.1.6.8
d41363bc-a582-4e60-94b4-a2b3348cf6dd,01.2.2,0.435810,0,11.1.1.2,0.336935,1,11.1.1.2
7383f6ab-af6f-433d-bf55-f021c9bd5a9b,01.1.3.1,0.488417,1,01.1.3.3,0.364777,0,01.1.3.1
b019c4d2-02d6-42d8-8525-397d6847814f,03.2.1.3,0.496750,1,03.2.1,0.364777,0,03.2.1.3
07f13fd9-99bf-4eab-909d-71710858abe8,11.1.1.1,0.346524,1,01.1.1.3,0.228521,0,11.1.1.1


## 5. Effet sur l'ensemble du test (routage + modèle)

- **Ancien** : argmax de l'ancien modèle sur tous les produits (comportement avant routage).
- **Nouveau** : candidat unique retenu tel quel (`route_and_score`), nouveau modèle ailleurs.

Sur les candidats uniques les deux donnent le même code : l'écart vient entièrement des produits multi.

In [15]:
test_all = table[table["split"] == "test"].reset_index(drop=True)
verite = test_all.drop_duplicates("id").set_index("id")
verite = merged.set_index("id").loc[verite.index, "code_lvl4"].astype("string").str.strip()

ancien_all = pick_best(test_all, score(ancien_eval, test_all[FEATURES])).set_index("id")
nouveau_all, _ = route_and_score(nouveau_eval, test_all)
nouveau_all = nouveau_all.set_index("id")

def acc(dec, ids):
    return float((dec.loc[ids, "sirus_code"].astype("string") == verite.loc[ids]).mean())

n_par_id = test_all.groupby("id").size()
groupes = {"tous les produits test": n_par_id.index,
           "candidat unique (routés)": n_par_id[n_par_id == 1].index,
           "≥ 2 candidats": n_par_id[n_par_id >= 2].index}
global_ = pd.DataFrame({g: {"produits": len(ids), "ancien": acc(ancien_all, ids), "nouveau": acc(nouveau_all, ids)}
                        for g, ids in groupes.items()}).T
global_["écart"] = global_["nouveau"] - global_["ancien"]
print("répartition des routes (nouveau) :", nouveau_all["sirus_route"].value_counts().to_dict())
global_.round(4)

répartition des routes (nouveau) :

 {'modele': 1662, 'candidat_unique': 1277}


,produits,ancien,nouveau,écart
tous les produits test,2939.0,0.8023,0.7931,-0.0092
candidat unique (routés),1277.0,0.9718,0.9718,0.0000
≥ 2 candidats,1662.0,0.6721,0.6558,-0.0162


## 6. Validation hors run : `2026-07-30/codif-x98xl`

Les sections précédentes testent sur le même run que l'entraînement. Ici, les deux modèles **livrés**
(ajustés sur 100 % de codif-c9vjm) scorent un autre run étiqueté, jamais vu : codif-x98xl (5 881 produits,
dont seuls ~6 % de couples libellé/code se retrouvent dans codif-c9vjm).

Ce run est antérieur au renommage des étapes : ses chemins diffèrent (`codif-lcs/`, `run-rag/`, `run-ttc/`,
`rag-annotation/`, `prune/`). Ses classifieurs sont aussi dans une version plus ancienne : un écart de niveau
entre les deux runs est donc attendu, seul l'écart **entre les deux modèles** compte.

In [16]:
ROOT_X = "s3://projet-budget-famille/data/workflow_runs/2026-07-30/codif-x98xl"   # LECTURE seule
merged_x = load_all_observations(
    lcs_path=f"{ROOT_X}/codif-lcs/raw_test_LCS.parquet",
    rag_path=f"{ROOT_X}/run-rag/predictions.parquet",
    ttc_path=f"{ROOT_X}/run-ttc/predictions.parquet",
    rag_annotations_path=f"{ROOT_X}/rag-annotation/predictions.parquet",
    mapping_path=f"{ROOT_X}/prune/mapping_lvl4.parquet",
)
table_x, _ = build_candidate_table(merged_x)
nx = table_x.groupby("id")["id"].transform("size")
multi_x = table_x[nx >= 2].reset_index(drop=True)
print(f"{table_x['id'].nunique()} produits, dont {multi_x['id'].nunique()} à ≥ 2 candidats")

def choix_argmax(rules, t):
    # Candidats de divisions inconnues écartés, comme en production.
    from src.scorer import scorable_mask
    t = t[scorable_mask(rules, t)].reset_index(drop=True)
    best = pick_best(t, score(rules, t[FEATURES]))
    return best.merge(t[["id", "code_candidat", "correcte"]], left_on=["id", "sirus_code"],
                      right_on=["id", "code_candidat"]).set_index("id")

cx = {"ancien": choix_argmax(ancien_final, multi_x), "nouveau": choix_argmax(nouveau_final, multi_x)}
ids = cx["ancien"].index.intersection(cx["nouveau"].index)
a, n = cx["ancien"].loc[ids, "correcte"], cx["nouveau"].loc[ids, "correcte"]
b01, b10 = int(((a == 0) & (n == 1)).sum()), int(((a == 1) & (n == 0)).sum())

from scipy.stats import binomtest
p_mcnemar = binomtest(b01, b01 + b10, 0.5).pvalue if b01 + b10 else float("nan")
upper_x = float(multi_x.groupby("id")["correcte"].max().loc[ids].mean())
pd.DataFrame({
    "produits multi scorés": [len(ids)] * 2,
    "accuracy produit": [a.mean(), n.mean()],
    "borne haute": [upper_x] * 2,
}, index=["ancien", "nouveau"]).round(4), f"nouveau corrige {b01}, dégrade {b10} — McNemar exact p = {p_mcnemar:.3f}"

5881 produits, dont 3847 à ≥ 2 candidats


(         produits multi scorés  accuracy produit  borne haute
 ancien                    3847            0.6875       0.8435
 nouveau                   3847            0.6998       0.8435,
 'nouveau corrige 152, dégrade 105 — McNemar exact p = 0.004')

In [17]:
# Fiabilité à volume égal, hors run.
cal_x = {nom: {"proba": list(d.loc[ids, "sirus_proba"]), "correct": list(d.loc[ids, "correcte"])} for nom, d in cx.items()}
vx = pd.DataFrame({nom: fiabilite_par_volume(c) for nom, c in cal_x.items()})
vx["écart"] = vx["nouveau"] - vx["ancien"]
vx.index.name = "volume codé automatiquement"
vx.round(4)

,ancien,nouveau,écart
volume codé automatiquement,,,
10%,0.9506,0.9766,0.0260
20%,0.9467,0.9181,-0.0286
30%,0.9376,0.9307,-0.0069
40%,0.9175,0.9051,-0.0123
50%,0.8784,0.8877,0.0094
60%,0.8332,0.8549,0.0217
70%,0.7961,0.8073,0.0111
80%,0.7544,0.7898,0.0354
90%,0.7207,0.7522,0.0315


## 7. Règles retenues

Modèles **livrés** (ajustés sur 100 % de leur population). Lecture d'une règle :
`if <condition> then <sortie si vraie> else <sortie si fausse>` ; la probabilité d'un candidat est la
moyenne non pondérée des sorties des 20 règles.

In [18]:
print(f"=== NOUVEAU modèle (multi-candidats, {len(nouveau_final.rules)} règles) ===")
print((OUT / "rules_printed.txt").read_text(encoding="utf-8"))
print(f"=== ANCIEN modèle (toutes lignes, {len(ancien_final.rules)} règles) ===")
print((ancien_dir / "rules_printed.txt").read_text(encoding="utf-8"))

=== NOUVEAU modèle (multi-candidats, 20 règles) ===
 [1] "Proportion of class 1 = 0.326 - Sample size n = 21244"                            
 [2] "if nb_votants < 2 then 0.138 (n=14436) else 0.725 (n=6808)"                       
 [3] "if conf_ragann < 0.95 then 0.212 (n=16917) else 0.773 (n=4327)"                   
 [4] "if vote_ragann < 1 then 0.166 (n=13824) else 0.625 (n=7420)"                      
 [5] "if nb_votants >= 2 & conf_ragann >= 0.95 then 0.87 (n=3446) else 0.221 (n=17798)" 
 [6] "if vote_ragann >= 1 & nb_votants >= 2 then 0.809 (n=4799) else 0.185 (n=16445)"   
 [7] "if conf_ttc < 0.92 then 0.219 (n=17005) else 0.754 (n=4239)"                      
 [8] "if vote_lcs < 1 & nb_votants < 2 then 0.207 (n=9211) else 0.417 (n=12033)"        
 [9] "if nb_votants < 2 & dist_lcs >= 1.5 then 0.207 (n=9211) else 0.417 (n=12033)"     
[10] "if vote_ragann < 1 & conf_ttc >= 0.92 then 0.557 (n=1938) else 0.303 (n=19306)"   
[11] "if conf_rag < 0.95 then 0.25 (n=18008) else 0.748 (n

In [19]:
def regles(path):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    lignes = []
    for r in payload["rules"]:
        cond = " & ".join(
            f"{c['var']} {c['op']} {c.get('value', c.get('levels'))}" for c in r["conditions"]
        )
        lignes.append({"condition": cond, "si_vraie": float(r["outputs"][0]), "sinon": float(r["outputs"][1]),
                       "n_vraie": r["supp_size"][0], "n_fausse": r["supp_size"][1], "fréquence": r["frequency"]})
    return pd.DataFrame(lignes)

r_nouveau, r_ancien = regles(OUT / "rules.json"), regles(ancien_dir / "rules.json")
communes = set(r_nouveau["condition"]) & set(r_ancien["condition"])
print(f"{len(communes)} condition(s) communes aux deux modèles sur {len(r_nouveau)} / {len(r_ancien)}")
r_nouveau.assign(aussi_dans_ancien=r_nouveau["condition"].isin(communes)).round(3)

6 condition(s) communes aux deux modèles sur 20 / 20


,condition,si_vraie,sinon,n_vraie,n_fausse,fréquence,aussi_dans_ancien
0,nb_votants < 2,0.138,0.725,14436,6808,0.305,True
1,conf_ragann < 0.95,0.212,0.773,16917,4327,0.199,False
2,vote_ragann < 1,0.166,0.625,13824,7420,0.172,True
3,nb_votants >= 2 & conf_ragann >= 0.95,0.870,0.221,3446,17798,0.148,False
4,vote_ragann >= 1 & nb_votants >= 2,0.809,0.185,4799,16445,0.123,True
5,conf_ttc < 0.920000016689301,0.219,0.754,17005,4239,0.119,False
6,vote_lcs < 1 & nb_votants < 2,0.207,0.417,9211,12033,0.095,True
7,nb_votants < 2 & dist_lcs >= 1.5,0.207,0.417,9211,12033,0.091,False
8,vote_ragann < 1 & conf_ttc >= 0.920000016689301,0.557,0.303,1938,19306,0.086,False
9,conf_rag < 0.95,0.250,0.748,18008,3236,0.079,False


## 8. Conclusion et limites

- **Le filtre multi-candidats n'est pas retenu, faute de résultat robuste.** Sur les produits où SIRUS
  tranche, il perd 1,6 point sur le test 20 % de codif-c9vjm (section 4, p ≈ 0,02) mais en gagne 1,2 sur
  codif-x98xl hors échantillon (section 6, p ≈ 0,004). Les deux mesures ne sont pas symétriques : la
  première compare des modèles à 80 % sur des classifieurs de la même version, la seconde des modèles à 100 %
  sur des classifieurs plus anciens. `build-table` garde donc toutes les lignes par défaut, et le modèle en
  production reste le même. Le filtre reste disponible (`--multi-candidates-only`) pour trancher sur un run
  étiqueté futur, produit par les classifieurs actuels.
- **Le routage, lui, est retenu** avec le modèle actuel (`argo/params.yaml` inchangé) : même code sur les
  candidats uniques (section 5), mais plus de score publié là où il n'y avait rien à choisir.
  `sirus_proba` y vaut NaN (`sirus_route = "candidat_unique"`) : un seuil d'exploitation appliqué en aval doit
  traiter cette route explicitement (a priori : code automatique).
- **Deux runs seulement**, et le second tourne avec des classifieurs plus anciens. Rien n'a été loggué sur
  MLflow ni écrit sur S3.